# Import repaired BAFU EcoSpold files

Create or reuse a local Brightway project with **ecoinvent biosphere 3.10**, then try importing the repaired EcoSpold 1 files. Select the **bw** kernel and run the cells from top to bottom.

The repaired files must already exist. To generate them, run this from the repository root:

```bash
conda run --no-capture-output -n bw python "scripts/ecospold importer/repair_all.py"
```

Project storage is `artifacts/brightway/` (ignored by Git). The first project setup downloads Brightway's biosphere archive and requires internet access.

In [4]:
import os
import sys
from pathlib import Path


ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts/ecospold importer").is_dir()
)
SOURCE = ROOT / "data/processed/ecospold1-schema-fixed"
STORAGE = ROOT / "artifacts/brightway"

PROJECT = "bafu-2026-biosphere-310"
BIOSPHERE = "ecoinvent-3.10-biosphere"
DATABASE = "BAFU:2026"

# Configure storage and the local helper path before importing Brightway.
STORAGE.mkdir(parents=True, exist_ok=True)
os.environ["BRIGHTWAY2_DIR"] = str(STORAGE)
sys.path.insert(0, str(ROOT / "scripts/ecospold importer"))

In [5]:
# Run the setup cell above first.
import bw2data as bd
import bw2io as bi
from date_compat import xml_date_parser
from timestamp_compat import iso_timestamp_parser

## Create the project

Reuse the same project as the Python import script, or create it from Brightway's biosphere 3.10 archive on first use.

In [6]:
if PROJECT not in bd.projects:
    bi.install_project("ecoinvent-3.10-biosphere", project_name=PROJECT)
    
bd.projects.set_current(PROJECT)

Restoring project backup archive - this could take a few minutes...
Restored project: bafu-2026-biosphere-310
13:52:08+0200 [info     ] Applying automatic update: 4.0 database search directories FTS5
13:52:08+0200 [info     ] Reindexing database ecoinvent-3.10-biosphere
13:52:08+0200 [info     ] Applying automatic update: 4.7 database dependencies in datapackage
13:52:08+0200 [info     ] Updating all LCIA methods     


668it [00:17, 37.59it/s] 

13:52:26+0200 [info     ] Updating all LCI databases    



1it [00:00, 203.24it/s]


In [8]:
bd.databases

Databases dictionary with 1 object(s):
	ecoinvent-3.10-biosphere

## Extract the repaired files

Use the local date/timestamp adapters and the standard EcoSpold 1 importer strategies. The XML files remain unchanged. The `importer` object stays available for inspection.

In [21]:
with iso_timestamp_parser(), xml_date_parser():
    importer = bi.SingleOutputEcospold1Importer(str(SOURCE), DATABASE, use_mp=False)

importer.apply_strategies()

  1%|▋                                                                            | 108/11947 [00:00<00:21, 544.13it/s]/opt/homebrew/Caskroom/miniforge/base/envs/bw/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:408: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
100%|███████████████████████████████████████████████████████████████████████████| 11947/11947 [00:23<00:00, 506.33it/s]


Extracted 11947 datasets in 23.65 seconds
Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 9 strategies in 1.01 seconds


In [22]:
importer.match_database(fields=["name", "reference product", "location"])

Applying strategy: link_iterable_by_fields


In [23]:
importer.match_database("ecoinvent-3.10-biosphere",fields=["name", "categories", "unit"])

Applying strategy: link_iterable_by_fields


In [24]:
importer.statistics()

Graph statistics for `BAFU:2026` importer:
11947 graph nodes:
	process: 11947
420063 graph edges:
	biosphere: 293747
	technosphere: 114369
	production: 11947
124850 edges to the following databases:
	BAFU:2026: 124850
2703 unique unlinked edges (295213 total):
	technosphere: 24
	biosphere: 2679




(11947, 420063, 295213, 0)

In [26]:
importer.data[0]

{'tags': [('ecoSpold01datasetRelatesToProduct', True),
  ('ecoSpold01infrastructureProcess', False),
  ('ecoSpold01infrastructureIncluded', False),
  ('ecoSpold01localName',
   'xx Cellulose fibre, inclusive blowing in, at plant'),
  ('ecoSpold01localCategory', 'material, obsolete'),
  ('ecoSpold01localSubCategory',
   'construction, obsolete\\insulation, obsolete'),
  ('ecoSpold01category', 'material, obsolete'),
  ('ecoSpold01subCategory', 'construction, obsolete\\insulation, obsolete'),
  ('ecoSpold01includedProcesses',
   'includes the input energy and material to the production processes, transports of the materials and the available process emissions. Also included is the packaging and the energy for the application of the fibres as insulation materi-als in buildings.'),
  ('ecoSpold01dataValidForEntirePeriod', True),
  ('ecoSpold01endDate', '2000-12-31'),
  ('ecoSpold01startDate', '2000-01-01'),
  ('ecoSpold01type', 1),
  ('ecoSpold01impactAssessmentResult', False),
  ('ecoSpold

In [25]:
for u in importer.unlinked:
    if u["type"] == "technosphere":
        print(u)

{'categories': ('electricity', 'production mix'), 'location': '', 'unit': 'kilowatt hour', 'name': 'Electricity, medium voltage, production ENTSO-E, at grid', 'type': 'technosphere', 'infrastructureProcess': False, 'comment': '(3,5,5,1,3,5); Data estimated from related processes\n', 'uncertainty type': 2, 'amount': 0.00683, 'loc': -4.9864306053994385, 'scale': 0.26236426446749106, 'negative': False}
{'categories': ('electricity', 'production mix'), 'location': '', 'unit': 'kilowatt hour', 'name': 'Electricity, low voltage, production ENTSO-E, at grid', 'type': 'technosphere', 'infrastructureProcess': False, 'comment': '(1,3,2,1,1,4); average value\n', 'uncertainty type': 2, 'amount': 0.207, 'loc': -1.575036485716768, 'scale': 0.061108816362124695, 'negative': False}
{'categories': ('electricity', 'production mix'), 'location': '', 'unit': 'kilowatt hour', 'name': 'Electricity, high voltage, production ENTSO-E, at grid', 'type': 'technosphere', 'infrastructureProcess': False, 'comment':

In [13]:
importer.drop_unlinked(i_am_reckless=True)

Applying strategy: drop_unlinked
Applied 1 strategies in 0.13 seconds


In [14]:
importer.write_database()

14:00:36+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████████████████████████████████████████████████████████████████████| 11947/11947 [00:02<00:00, 5602.40it/s]


14:00:38+0200 [info     ] Vacuuming database            
Created database: BAFU:2026


Brightway2 SQLiteBackend: BAFU:2026

In [19]:
import bw2calc as bc
method = bd.methods.random()
act = bd.Database("BAFU:2026").random()
lca = bc.LCA({act: 1}, method)
lca.lci()
lca.lcia()
print(lca.score)

EmptyBiosphere: 

In [20]:
act.as_dict()

{'tags': [('ecoSpold01datasetRelatesToProduct', True),
  ('ecoSpold01infrastructureProcess', False),
  ('ecoSpold01infrastructureIncluded', False),
  ('ecoSpold01localName',
   'Transport, passenger helicopter, twin-engine, Fahrzeug'),
  ('ecoSpold01localCategory', 'transport systems'),
  ('ecoSpold01localSubCategory', 'helicopter'),
  ('ecoSpold01category', 'transport systems'),
  ('ecoSpold01subCategory', 'helicopter'),
  ('ecoSpold01includedProcesses',
   'This dataset describes the demand of helicopter, demand of LTOs, energy requirements and emissions of one hour helicopter flight.'),
  ('ecoSpold01dataValidForEntirePeriod', True),
  ('ecoSpold01endDate', '2015-12-31'),
  ('ecoSpold01startDate', '2009-01-01'),
  ('ecoSpold01type', 1),
  ('ecoSpold01impactAssessmentResult', False),
  ('ecoSpold01version', '2023'),
  ('ecoSpold01internalVersion', '0.0'),
  ('ecoSpold01timestamp', '2017-04-04T00:00:00+02:00'),
  ('ecoSpold01languageCode', 'en'),
  ('ecoSpold01localLanguageCode', 'de'